from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd, sqlite3
con = sqlite3.connect("../data/social.db")
def q(sql): return pd.read_sql(sql, con)

SELECT COUNT(*), SUM(likes), AVG(likes), MIN(likes), MAX(likes)
FROM posts

AVG(comments) averages only the non-NULL rows
COUNT(comments) counts only the non-NULL rows
COUNT(*) counts every row

SELECT brand, AVG(likes) AS avg_likes
FROM posts
GROUP BY brand

df.groupby("brand")["likes"].mean()	
SELECT brand, AVG(likes) FROM posts GROUP BY brand

.agg(n=("post_id","count"), total=("likes","sum"))	
SELECT brand, COUNT(*) AS n, SUM(likes) AS total ... GROUP BY brand

groupby(["platform","post_type"])	
GROUP BY platform, post_type

SELECT brand,
       COUNT(*)            AS n_posts,
       SUM(likes)          AS total_likes,
       ROUND(AVG(likes),1) AS avg_likes,
       MAX(likes)          AS max_likes
FROM posts
GROUP BY brand
ORDER BY total_likes DESC

SELECT brand, AVG(likes) AS avg_likes
FROM posts
GROUP BY brand
HAVING AVG(likes) > 1000           -- keep only brands whose AVERAGE is over 1000

WHERE throws out individual papers before you sort them.
HAVING throws out whole piles after you’ve counted them.

SELECT brand, COUNT(*) AS n
FROM posts
WHERE platform = 'TikTok'          -- step 1: only TikTok rows
GROUP BY brand                      -- step 2: piles by brand
HAVING COUNT(*) >= 20               -- step 3: only piles with 20+ posts

In pandas, HAVING needed two steps (groupby, then a mask on the result). In SQL it’s one clause.

FROM: table
WHERE: filter rows
GROUP BY: make groups
HAVING: filter groups
SELECT: columns and aggregates
ORDER BY: sort
LIMIT: cut

SELECT post_id, likes,
       CASE
           WHEN likes >= 5000 THEN 'viral'
           WHEN likes >= 1000 THEN 'good'
           ELSE 'low'
       END AS tier
FROM posts

SELECT CASE WHEN likes >= 5000 THEN 'viral'
            WHEN likes >= 1000 THEN 'good'
            ELSE 'low' END AS tier,
       COUNT(*) AS n
FROM posts
GROUP BY tier

SELECT brand,
       COUNT(*)                                       AS n_posts,
       SUM(CASE WHEN sponsored = 1 THEN 1 ELSE 0 END) AS n_sponsored,
       SUM(CASE WHEN platform = 'TikTok' THEN likes ELSE 0 END) AS tiktok_likes
FROM posts
GROUP BY brand

## 3.2 Cheat sheet: GROUP BY, HAVING, CASE

| I want… | SQL |
|---|---|
| one stat overall | `SELECT AVG(x) FROM t` |
| stat per group | `SELECT g, AVG(x) FROM t GROUP BY g` |
| several stats | `SELECT g, COUNT(*) AS n, SUM(x) AS s ... GROUP BY g` |
| two grouping cols | `GROUP BY g1, g2` |
| filter rows first | `WHERE ...` (before GROUP BY) |
| filter groups | `HAVING COUNT(*) > 5` (after GROUP BY) |
| if/else label | `CASE WHEN c1 THEN 'a' WHEN c2 THEN 'b' ELSE 'c' END AS lbl` |
| count rows meeting a condition | `SUM(CASE WHEN cond THEN 1 ELSE 0 END)` |
| sum only some rows | `SUM(CASE WHEN cond THEN x ELSE 0 END)` |
| rate of a 0/1 column | `AVG(col)` |
| rate of a condition | `AVG(CASE WHEN cond THEN 1.0 ELSE 0 END)` |

**Rules**
- SELECT columns: in GROUP BY or inside an aggregate
- WHERE can't use aggregates → HAVING
- Aggregates skip NULL; COUNT(*) doesn't
- CASE: first true branch wins; ELSE catches NULL too
- Runs: FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT

In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import sqlite3

con = sqlite3.connect("../data/social.db")

tables = {
    "posts": "../data/social_posts.csv",
    "brands": "../data/brands.csv",
    "orders": "../data/gate_orders.csv",
    "customers": "../data/gate_customers.csv",
}
for name, path in tables.items():
    pd.read_csv(path).to_sql(name, con, index=False, if_exists="replace")

pd.DataFrame({
    "name":  ["a", "b", "c", "d", "e"],
    "team":  ["x", "y", "x", "y", "x"],
    "score": [5, 12, None, 20, 8],
    "city":  ["UK", "US", "UK", None, "US"],
}).to_sql("toy", con, index=False, if_exists="replace")

def q(sql):
    return pd.read_sql(sql, con)

q("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,posts
1,brands
2,orders
3,customers
4,toy


In [2]:
q("SELECT * FROM toy")

,name,team,score,city
0,a,x,5.0,UK
1,b,y,12.0,US
2,c,x,NaN,UK
3,d,y,20.0,NaN
4,e,x,8.0,US


In [3]:
q("SELECT team, COUNT(*) FROM toy GROUP BY team")
# x--3, y--2
q("SELECT team, COUNT(score) FROM toy GROUP BY team")
# x-- 2, y--2
q("SELECT team, AVG(score) FROM toy GROUP BY team")
# x--6.5 y-- 16
q("SELECT team, SUM(score) FROM toy GROUP BY team")
# x--13, y--32
q("SELECT team FROM toy GROUP BY team HAVING COUNT(*) > 2")
# x
q("SELECT team, COUNT(*) FROM toy WHERE score > 6 GROUP BY team")
# x--1, y--2
q("SELECT CASE WHEN score >= 10 THEN 'high' ELSE 'low' END AS lvl, COUNT(*) FROM toy GROUP BY lvl") 
#(careful: where does c go?)
# high-- 2, low-- 3 
# c’s score is NULL, and NULL >= 10 isn’t true, so it falls through to ELSE, i.e. “low”. COUNT(*) just counts it once it’s there.c will be counted because count(*)
q("SELECT team, SUM(CASE WHEN city = 'UK' THEN 1 ELSE 0 END) FROM toy GROUP BY team")
# x-- 2, y-- 0
q("SELECT team, MAX(score) - MIN(score) FROM toy GROUP BY team")
# x-- 3, y-- 8
q("SELECT team, AVG(score) FROM toy GROUP BY team HAVING AVG(score) > 10")
#what happens and why? aggs cannot be put inside where, having filters grouped results
# y--16

,team,COUNT(*)
0,x,3
1,y,2


,team,COUNT(score)
0,x,2
1,y,2


,team,AVG(score)
0,x,6.5
1,y,16.0


,team,SUM(score)
0,x,13.0
1,y,32.0


,team
0,x


,team,COUNT(*)
0,x,1
1,y,2


,lvl,COUNT(*)
0,high,2
1,low,3


,team,SUM(CASE WHEN city = 'UK' THEN 1 ELSE 0 END)
0,x,2
1,y,0


,team,MAX(score) - MIN(score)
0,x,3.0
1,y,8.0


,team,AVG(score)
0,y,16.0


In [4]:
# B1. Posts per platform

# (a) Count posts per platform, as column n.
# (b) Sort highest first.

# FROM: table
# WHERE: filter rows
# GROUP BY: make groups
# HAVING: filter groups
# SELECT: columns and aggregates
# ORDER BY: sort
# LIMIT: cut

q("SELECT platform, COUNT(*) AS n FROM posts GROUP BY platform ORDER BY n DESC")

,platform,n
0,Instagram,132
1,TikTok,104
2,Facebook,64


In [5]:
# B2. Brand report

# (a) Per brand:
# n_posts (count)
# total_likes (sum)
# avg_likes (average, rounded to 1)
# max_likes
# (b) Sort by total_likes, highest first.

q("""
SELECT brand, 
    COUNT(*) AS n_posts,
    SUM(likes) AS total_likes,
    ROUND(AVG(likes),1) AS avg_likes,
    MAX(likes) AS max_likes
FROM posts 
GROUP BY brand 
ORDER BY total_likes DESC
""")

,brand,n_posts,total_likes,avg_likes,max_likes
0,SnackBox,60,161327,2688.8,13577
1,Glowly,62,101657,1639.6,5761
2,FitFuel,56,66044,1179.4,4970
3,UrbanThread,56,47147,841.9,3384
4,HomeNest,66,30309,459.2,2032


In [6]:
# B3. HAVING

# (a) Brands with average likes over 1000: show brand and avg (rounded to 1).
# (b) Sort highest first.
# (c) Comment: why can’t this use WHERE?
# WHERE runs before grouping, so it can’t use AVG
q("""
SELECT brand, ROUND(AVG(likes), 1) AS avg 
FROM posts 
GROUP BY brand 
HAVING AVG(likes)>1000 
ORDER BY avg DESC
""")

,brand,avg
0,SnackBox,2688.8
1,Glowly,1639.6
2,FitFuel,1179.4


In [7]:
# B4. WHERE + HAVING together

# (a) Only TikTok posts.
# (b) Count per brand.
# (c) Keep brands with at least 20 TikTok posts.
# (d) Comment: which brand dropped out, and how many TikTok posts did it have?
q("""
SELECT brand, COUNT (*) as count
FROM posts 
WHERE platform= 'TikTok' 
GROUP BY brand
HAVING COUNT (*)>=20
""")
#Glowly-- 17 posts 

,brand,count
0,FitFuel,21
1,HomeNest,20
2,SnackBox,23
3,UrbanThread,23


In [8]:
# B5. Two keys

# (a) Average likes per platform and post_type, rounded to 1, plus a count n.
# (b) Top 3 combinations by average likes.
q("""
SELECT platform, post_type, ROUND(AVG(likes), 1) AS avg, COUNT(*) AS n
FROM posts 
GROUP BY platform, post_type
ORDER BY avg DESC
LIMIT 3
""")

,platform,post_type,avg,n
0,TikTok,video,2913.0,34
1,TikTok,carousel,2445.9,27
2,Instagram,video,1573.5,52


In [9]:
# B6. CASE WHEN tiers

# (a) Label each post 'viral' (likes ≥ 5000), 'good' (≥ 1000) or 'low'.
# (b) Count posts per tier.
# (c) Comment: what would go wrong if you checked >= 1000 before >= 5000?
#everything will be categorized in good before the 5000 check could run
q("""
SELECT COUNT(*), CASE 
    WHEN likes  >= 5000 THEN 'viral'
    WHEN likes  >= 1000 THEN 'good'
    ELSE 'low' 
END AS tier
FROM posts
GROUP BY tier
""")

,COUNT(*),tier
0,129,good
1,160,low
2,11,viral


In [10]:
# B7. Conditional aggregation

# (a) Per brand:
# n_posts
# n_sponsored using SUM(CASE WHEN …)
# pct_sponsored using AVG(sponsored) * 100, rounded to 1
# (b) Sort by pct_sponsored, highest first.
q("""
SELECT brand,
COUNT(*) AS n_posts,
SUM(CASE WHEN sponsored= 1 THEN 1 ELSE 0 END) AS n_sponsored,
ROUND(AVG(sponsored)*100.0, 1) AS pct_sponsored
FROM posts
GROUP BY brand
ORDER BY pct_sponsored DESC
""")

,brand,n_posts,n_sponsored,pct_sponsored
0,HomeNest,66,19,28.8
1,FitFuel,56,16,28.6
2,UrbanThread,56,14,25.0
3,SnackBox,60,13,21.7
4,Glowly,62,11,17.7


In [11]:
# B8. TikTok dependence

# (a) Per brand:
# tiktok_likes (likes from TikTok posts only, using SUM(CASE WHEN …))
# total_likes
# pct_tiktok = tiktok_likes × 100 / total_likes, rounded to 1 (watch the division!)
# (b) Sort by pct_tiktok, highest first.
# (c) Comment: which brand relies most on TikTok?

q("""
SELECT brand, 
    SUM (CASE WHEN platform = 'TikTok' THEN likes ELSE 0 END) AS tiktok_likes,
    SUM(likes) AS total_likes,
    ROUND(SUM(CASE WHEN platform = 'TikTok' THEN likes ELSE 0 END) * 100.0/ SUM(likes), 1) AS pct_tiktok
FROM posts
GROUP BY brand
ORDER BY pct_tiktok DESC
""")
# UrbanThread

,brand,tiktok_likes,total_likes,pct_tiktok
0,UrbanThread,30496,47147,64.7
1,SnackBox,99436,161327,61.6
2,FitFuel,34237,66044,51.8
3,HomeNest,13963,30309,46.1
4,Glowly,42595,101657,41.9


In [12]:
# B9. Orders table

# (a) Count orders per status in the orders table.
# (b) Comment: does this match what you found in the Phase 2 gate?
q("""
SELECT status, COUNT(*)
FROM orders
GROUP BY status
""")
#they match

,status,COUNT(*)
0,cancelled,37
1,delivered,340
2,returned,43


In [13]:
q("SELECT brand, AVG(likes) FROM posts GROUP BY brand HAVING AVG(likes) > 1000")
#where doesnt work with aggs

,brand,AVG(likes)
0,FitFuel,1179.357143
1,Glowly,1639.629032
2,SnackBox,2688.783333


In [14]:
q("SELECT brand,  COUNT(*) FROM posts GROUP BY brand ")
#selecting post_id will contradict the counting required

,brand,COUNT(*)
0,FitFuel,56
1,Glowly,62
2,HomeNest,66
3,SnackBox,60
4,UrbanThread,56


In [15]:
q("""SELECT CASE WHEN likes >= 5000 THEN 'viral'
            WHEN likes >= 1000 THEN 'good'
            ELSE 'low' END AS tier, 
COUNT(*) FROM posts 
GROUP BY tier""")

,tier,COUNT(*)
0,good,129
1,low,160
2,viral,11


In [16]:
# per plat
# a) n_posts
# (b) n_viral = posts with likes ≥ 5000 (SUM(CASE…))
# (c) pct_viral = % of posts that are viral, rounded to 2 (watch the division)
# (d) Sort by pct_viral, highest first.

q("""
SELECT platform,
    COUNT (*) AS n,
    SUM(CASE WHEN likes >= 5000 THEN 1 ELSE 0 END) AS n_viral,
    ROUND((SUM(CASE WHEN likes >= 5000 THEN 1.0 ELSE 0.0 END)/COUNT (*)) *100.0, 2) AS pct_viral
FROM posts
GROUP BY platform
ORDER BY pct_viral DESC
""")

,platform,n,n_viral,pct_viral
0,TikTok,104,8,7.69
1,Instagram,132,3,2.27
2,Facebook,64,0,0.00


In [17]:
# Q5. Write.
q("""
SELECT brand,
       ROUND(AVG(likes), 1) AS avg_spons_likes
FROM posts
WHERE sponsored = 1              -- 1. keep only sponsored rows
GROUP BY brand                   -- 2. then make piles by brand
HAVING AVG(likes) > 1500         -- 3. then keep piles whose average is over 1500
ORDER BY avg_spons_likes DESC
""")


# (a) Sponsored posts only.
# (b) Average likes per brand, rounded to 1.
# (c) Keep only brands where that average is over 1500 (write the aggregate in HAVING, not the alias).

,brand,avg_spons_likes
0,Glowly,1915.1
1,SnackBox,1792.0
